In [40]:
# IMPORTS

# 1. Symbolic Mathematics (Analytical Derivations & Units)
import sympy as sp
from IPython.display import display, Math # For rendering LaTeX beautifully in Jupyter

# 2. Numerical Arrays and Matrices
import numpy as np

# 3. Scientific Computing (Integration and Eigenvalue Solvers)
from scipy.integrate import quad
from scipy.linalg import eigh

# 4. Visualization (Convergence and Error Graphs)
import matplotlib.pyplot as plt

In [41]:
# CONFIGURATION, CONSTANTS & FLAGS

# --- Physical Parameters ---
# We parameterize the physical constants so the equations remain exact.
# They default to 1.0 (Natural Units) to prevent floating-point underflow.
L = 10.0      # The domain size of the infinite well [-L/2, L/2]
HBAR = 1.0    # Reduced Planck's constant
MASS = 1.0    # Particle mass
OMEGA = 1.0   # Oscillator frequency

# --- Variational Parameters ---
N_BASIS = 50  # The size of the basis (k = 1, 2, ..., N_BASIS)
N_STATES = 5  # How many low-lying energy levels to track (should be << N_BASIS)

# --- Execution Flags ---
# These control the logic flow, making debugging much easier.
DEBUG_PRINT = True       # Set to True to print matrix values (keep N_BASIS small if True!)
PLOT_ERROR = True        # Flag to trigger the matplotlib convergence graph
PLOT_WAVEFUNCTION = True # Flag to trigger visualizing the final wavefunctions

# --- Numerical Methods ---
integration_method = 'trapezoidal' # Options: 'analytical', 'scipy', 'trapezoidal', 'romberg', 'simpsons'
eigenvalue_method = 'scipy'        # Options: 'scipy', 'QR', 'Jacobi'
N_GRID = 1000                      # Number of grid intervals for custom integration methods

In [42]:
def basis_function(x, n, L) -> float:
    """
    Evaluates the n-th infinite square well basis function at position x.

    Parameters
    ----------
    x : float
        Position where the function is evaluated.
    n : int
        Basis index.
    L : float
        Width of the well.

    Returns
    -------
    float
        Value of the basis function at x.
    """
    normalization = np.sqrt(2.0 / L)
    argument = (n * np.pi * (x + L / 2.0)) / L
    
    return normalization * np.sin(argument)

def integrand_V(x, n, m, L, mass, omega) -> float:
    """
    The potential energy integrand: chi_n(x) * V(x) * chi_m(x).
    """
    chi_n = basis_function(x, n, L)
    chi_m = basis_function(x, m, L)
    
    V_x = 0.5 * mass * ((omega * x)**2)  # QHO Potential: 1/2 * mass * omega^2 * x^2
    
    return chi_n * V_x * chi_m

In [43]:
# NUMERICAL INTEGRATION METHODS

def integral_solve_using_trapezoidal_rule(f, a, b, N, *args) -> float:
    """
    Solves the definite integral of a function using the trapezoidal rule.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. Must accept the integration variable
        as its first argument, followed by any additional arguments (*args).
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The number of sub-intervals to divide the integration range into.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^2).
    """
    h = (b - a) / N
    s = f(a, *args) / 2.0
    
    for i in range(1, N):
        s += f(a + h * i, *args)
        
    s += f(b, *args) / 2.0
    s *= h
    
    return s


def integral_solve_using_romberg_method(f, a, b, N, *args) -> float:
    """
    Solves the definite integral of a function using Romberg's method.
    
    This method extrapolates a more accurate estimate from two trapezoidal 
    approximations, significantly reducing the approximation error.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. Must accept the integration variable
        as its first argument, followed by any additional arguments (*args).
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The initial number of sub-intervals. The method will also evaluate 
        the function at 2*N intervals internally.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^4).
    """
    # Calculate S_n using the trapezoidal rule
    s_n = integral_solve_using_trapezoidal_rule(f, a, b, N, *args)

    # Calculate S_2n using the trapezoidal rule with double the intervals
    s_2n = integral_solve_using_trapezoidal_rule(f, a, b, 2 * N, *args)

    # Romberg extrapolation formula
    solution = (4.0 / 3.0) * s_2n - (1.0 / 3.0) * s_n
    
    return solution

def integral_solve_using_simpsons_rule(f, a, b, N, *args) -> float:
    """
    Solves the definite integral of a function using Simpson's 1/3 rule.

    Parameters
    ----------
    f : callable
        The objective function to be integrated. 
    a : float
        The lower limit of integration.
    b : float
        The upper limit of integration.
    N : int
        The number of sub-intervals. If an odd integer is provided, 
        it will be automatically incremented to the next even integer.
    *args : tuple, optional
        Additional arguments to pass to the objective function `f`.

    Returns
    -------
    float
        The estimated numerical value of the integral. The error scales as O(h^4).
    """
    # Force N to be an integer first, just in case a float was passed
    N = int(N)
    
    # Automatically adjust odd numbers to the nearest even number (rounding up)
    if N % 2 != 0:
        N += 1
        print(f"Warning: Simpson's rule requires an even N. Automatically adjusted N to {N}.")

    h = (b - a) / N
    s = f(a, *args) + f(b, *args)

    # Sum the odd-indexed terms (multiplied by 4)
    for i in range(1, N, 2):
        s += 4.0 * f(a + i * h, *args)
        
    # Sum the even-indexed terms (multiplied by 2)
    for i in range(2, N - 1, 2):
        s += 2.0 * f(a + i * h, *args)

    return (h / 3.0) * s

In [44]:
# Calculate the Hamiltonian matrix elements (solve many integrals)

H = np.zeros((N_BASIS, N_BASIS))

# Hamiltonian is hermitian, so H_i,j = H_j,i*. We only need to calculate N_BASIS + (N_BASIS - 1) + (N_BASIS - 2) + ... + 1 = N_BASIS * (N_BASIS + 1) / 2 unique elements. The rest can be filled in by symmetry.

# H = T + V, where T is the kinetic energy term and V is the potential energy term.
for i in range(N_BASIS):
    for j in range(i, N_BASIS):

        # shift indices to match basis functions (important!)
        n = i + 1
        m = j + 1

        # --- Kinetic Energy (Exact analytical result) ---
        # Using T = (hbar^2 * k^2) / 2m
        if n == m:
            k = (n * np.pi) / L
            T = ((HBAR * k)**2) / (2.0 * MASS)
        else:
            T = 0.0
        
        # --- Potential Energy (Numerical Integration) ---
        V = 0.0

        if integration_method == 'analytical':
            pass #TODO
            # Use the analytical formula derived using SymPy
            # V = calculate_V_analytical(n, m, L) #TODO

        elif integration_method == 'scipy':
            V, _ = quad(integrand_V, -L/2, L/2, args=(n, m, L, MASS, OMEGA), limit=100)

        elif integration_method == 'trapezoidal':
            V = integral_solve_using_trapezoidal_rule(integrand_V, -L/2, L/2,N_GRID, n, m, L, MASS, OMEGA)

        elif integration_method == 'romberg':
            V = integral_solve_using_romberg_method(integrand_V, -L/2, L/2, N_GRID, n, m, L, MASS, OMEGA)

        elif integration_method == 'simpsons':
            V = integral_solve_using_simpsons_rule(integrand_V, -L/2, L/2, N_GRID, n, m, L, MASS, OMEGA)

        else:
            raise ValueError("Invalid integration method selected!")
        
        H[i, j] = T + V
        if i != j:
            H[j, i] = H[i, j]  # Fill in the symmetric element (base functions are real, so H is symmetric)

In [45]:
# Maybe calculate S matrix here for generality, but for the infinite square well, the basis functions are orthonormal, so S = I (identity matrix).

In [46]:
# Solve secular equation, which is just the eigenvalue problem for the Hamiltonian matrix H.

if eigenvalue_method == 'scipy':
    # Use SciPy's eigh function to solve the eigenvalue problem
    energies, wavefunctions = eigh(H)

elif eigenvalue_method == 'QR':
    # Use a custom QR algorithm to solve the eigenvalue problem
    energies, wavefunctions = custom_qr_algorithm(H)

elif eigenvalue_method == 'Jacobi':
    # Use a custom Jacobi iteration to solve the eigenvalue problem
    energies, wavefunctions = custom_jacobi_iteration(H)

else:
    raise ValueError("Invalid eigenvalue method selected!")

In [48]:
energies

array([  0.5       ,   1.5       ,   2.50000008,   3.50000122,
         4.50001264,   5.50009872,   6.50060236,   7.502928  ,
         8.51147133,   9.53657299,  10.59619164,  11.71311527,
        12.90851533,  14.19698614,  15.58608706,  17.07864367,
        18.67512237,  20.37507232,  22.17779761,  24.08261419,
        26.08893161,  28.19626149,  30.40420996,  32.7124573 ,
        35.12074804,  37.62887166,  40.23666053,  42.9439719 ,
        45.75069419,  48.65672662,  51.66199492,  54.76642385,
        57.9699661 ,  61.27256039,  64.67418195,  68.17477341,
        71.77433408,  75.47279795,  79.27019961,  83.16644175,
        87.16162817,  91.25557656,  95.44856228,  99.7401616 ,
       104.13124858, 108.62045314, 113.20844991, 117.89522538,
       123.38194115, 128.25238084])